# Prepare Dataset

Converts the Amazon Reviews 2023 gzipped JSONL data (reviews + meta) into RecBole Atomic Files (.inter). 

In [1]:
import json
import gzip
from tqdm import tqdm
from pathlib import Path
from typing import Any
from collections.abc import Iterable
import pandas as pd

In [2]:
# --- Config ---
DATASET_NAME: str = "Beauty_and_Personal_Care"
CATEGORIES: list[str] = ["Beauty_and_Personal_Care"]
SAMPLE_SIZE: int | None = None            # None = all rows
DATA_DIR: str = "../data"

# Only include reviews that satisfy all of the following criteria
START_DATE: str | None = "2020-01-01"
END_DATE: str | None = "2022-12-31"
MIN_RATING: int | None = None

# Data split ratios
TRAIN_RATIO: float = 0.8
VALID_RATIO: float = 0.1

# Cold vs. warm users
WARM_USER_MIN_REVIEWS: int = 5

# Sequential recommendation config
MAX_ITEM_LIST_LENGTH: int = 50

## Common utils

In [3]:
def stream_jsonl(path: str, fields: list[str] | None = None):
    with gzip.open(path, "rt", encoding="utf-8") as f:
        for _, line in enumerate(f):
            obj = json.loads(line)
            if fields is not None:
                obj = {k: obj.get(k) for k in fields}
            yield obj

def _date_to_ms(date_str: str | None) -> int | None:
    if date_str is None:
        return None
    return int(pd.Timestamp(date_str, tz="UTC").timestamp() * 1000)

def load_reviews(
    categories: list[str], sample_size: int | None = None, 
    start_date: str | None = None, end_date: str | None = None, 
    min_rating: int | None = None
) -> list[dict[str, Any]]:
    start_ts = _date_to_ms(start_date)
    end_ts = _date_to_ms(end_date)
    print(f"Filtering reviews with criteria: start_date={start_date}, end_date={end_date}, min_rating={min_rating}")

    reviews: list[dict[str, Any]] = []
    for cat in categories:
        path = f"{DATA_DIR}/{cat}.jsonl.gz"
        print(f"Loading reviews: {path}")
        for obj in stream_jsonl(path, fields=[
            'user_id', 'parent_asin', 'rating', 'timestamp'
        ]):
            ts: Any = obj.get("timestamp")
            rating: Any = obj.get("rating")
            if start_ts is not None and (ts is not None and ts < start_ts):
                continue
            if end_ts is not None and (ts is not None and ts > end_ts):
                continue
            if min_rating is not None and (rating is not None and rating < min_rating):
                continue
            obj["category"] = cat
            reviews.append(obj)

            if sample_size is not None and len(reviews) >= sample_size:
                print(f"Reached sample size limit ({sample_size} reviews). Stopping.")
                break
    return reviews


## Load reviews

In [4]:
reviews = load_reviews(
    CATEGORIES, sample_size=SAMPLE_SIZE, start_date=START_DATE, end_date=END_DATE,
    min_rating=MIN_RATING
)
print(f"Loaded {len(reviews)} reviews")

Filtering reviews with criteria: start_date=2020-01-01, end_date=2022-12-31, min_rating=None
Loading reviews: ../data/Beauty_and_Personal_Care.jsonl.gz
Loaded 10947316 reviews


In [5]:
df_reviews = pd.DataFrame(reviews)
display(df_reviews.head())
df_reviews.info()

,user_id,parent_asin,rating,timestamp,category
0,AFKZENTNBQ7A7V7UXW5JJI6UGRYQ,B00Z03RC80,1.0,1616743454733,Beauty_and_Personal_Care
1,AFKZENTNBQ7A7V7UXW5JJI6UGRYQ,B085PRT2MP,1.0,1614915977684,Beauty_and_Personal_Care
2,AFKZENTNBQ7A7V7UXW5JJI6UGRYQ,B08G81QQ9L,5.0,1612052493701,Beauty_and_Personal_Care
3,AFKZENTNBQ7A7V7UXW5JJI6UGRYQ,B07YYG76X1,1.0,1609700981786,Beauty_and_Personal_Care
4,AFKZENTNBQ7A7V7UXW5JJI6UGRYQ,B07X4FKLNK,3.0,1581313195358,Beauty_and_Personal_Care


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10947316 entries, 0 to 10947315
Data columns (total 5 columns):
 #   Column       Dtype  
---  ------       -----  
 0   user_id      object 
 1   parent_asin  object 
 2   rating       float64
 3   timestamp    int64  
 4   category     object 
dtypes: float64(1), int64(1), object(3)
memory usage: 417.6+ MB


## Split train/valid/test with cutoff timestamps

In [6]:
timestamps = sorted(df_reviews["timestamp"].values)
train_end_ts = timestamps[int(len(timestamps) * TRAIN_RATIO)]
valid_end_ts = timestamps[int(len(timestamps) * (TRAIN_RATIO + VALID_RATIO))]
print(f"Train end timestamp: {train_end_ts} ({pd.Timestamp(train_end_ts, unit='ms', tz='UTC')})")
print(f"Valid end timestamp: {valid_end_ts} ({pd.Timestamp(valid_end_ts, unit='ms', tz='UTC')})")

Train end timestamp: 1654556501142 (2022-06-06 23:01:41.142000+00:00)
Valid end timestamp: 1663665351202 (2022-09-20 09:15:51.202000+00:00)


In [7]:
df_reviews_train = df_reviews[df_reviews["timestamp"] <= train_end_ts]
df_reviews_valid = df_reviews[(df_reviews["timestamp"] > train_end_ts) & (df_reviews["timestamp"] <= valid_end_ts)]
df_reviews_test = df_reviews[df_reviews["timestamp"] > valid_end_ts]
print(f"Train reviews: {len(df_reviews_train)}")
print(f"Valid reviews: {len(df_reviews_valid)}")
print(f"Test reviews: {len(df_reviews_test)}")

Train reviews: 8757853
Valid reviews: 1094732
Test reviews: 1094731


## Write .inter files (user-item interactions)

In [8]:
# As RecBole expects integer IDs, we need to create mappings from the original string IDs to integers.
user_ids: set[str] = set()
item_ids: set[str] = set()
for r in reviews:
    user_ids.add(r["user_id"])
    item_ids.add(r["parent_asin"])

user_map: dict[str, int] = {uid: i for i, uid in enumerate(sorted(user_ids))}
item_map: dict[str, int] = {pid: i for i, pid in enumerate(sorted(item_ids))}
print(f"Users: {len(user_map):,}  Items: {len(item_map):,}")

Users: 6,150,175  Items: 595,753


In [9]:
out_dir: Path = Path(DATA_DIR) / DATASET_NAME
prefix: Path = out_dir / DATASET_NAME

In [10]:
df_sorted: pd.DataFrame = pd.concat(
    [
        df_reviews_train.assign(split="train"),
        df_reviews_valid.assign(split="valid"),
        df_reviews_test.assign(split="test"),
    ],
    ignore_index=True,
).sort_values(
    ["user_id", "timestamp"],
    ascending=[True, True],
    kind="mergesort",
)

rows_by_split: dict[str, list[dict[str, str | int | float]]] = {
    "train": [],
    "valid": [],
    "test": [],
}

user_groups = df_sorted.groupby("user_id", sort=False)
for user_id, df_user in tqdm(
    user_groups,
    total=df_sorted["user_id"].nunique()
):
    history: list[int] = []

    for _, row in df_user.iterrows():
        uid: int | None = user_map.get(row["user_id"])
        iid: int | None = item_map.get(row["parent_asin"])
        if uid is None or iid is None:
            continue

        if history:
            split_name: str = str(row["split"])
            item_history: list[int] = history[-MAX_ITEM_LIST_LENGTH:]
            rows_by_split[split_name].append(
                {
                    "user_id": uid,
                    "item_id": iid,
                    "rating": float(row["rating"]),
                    "timestamp": float(row["timestamp"]),
                    "item_id_list": " ".join(str(item_id) for item_id in item_history),
                }
            )

        history.append(iid)

def write_inter_file(path: Path, rows: Iterable[dict[str, str | int | float]]) -> None:
    out_dir.mkdir(parents=True, exist_ok=True)
    row_count: int = 0

    with path.open("w", encoding="utf-8") as f:
        f.write(
            "user_id:token\t"
            "item_id:token\t"
            "rating:float\t"
            "timestamp:float\t"
            "item_id_list:token_seq\n"
        )

        for row in rows:
            f.write(
                f"{row['user_id']}\t"
                f"{row['item_id']}\t"
                f"{row['rating']}\t"
                f"{row['timestamp']}\t"
                f"{row['item_id_list']}\n"
            )
            row_count += 1

    print(f"Wrote {path} ({row_count:,} rows)")

write_inter_file(prefix.with_suffix(".train.inter"), rows_by_split["train"])
write_inter_file(prefix.with_suffix(".valid.inter"), rows_by_split["valid"])
write_inter_file(prefix.with_suffix(".test.inter"), rows_by_split["test"])

100%|██████████| 6150175/6150175 [08:14<00:00, 12435.58it/s]


Wrote ../data/Beauty_and_Personal_Care/Beauty_and_Personal_Care.train.inter (3,532,394 rows)
Wrote ../data/Beauty_and_Personal_Care/Beauty_and_Personal_Care.valid.inter (608,100 rows)
Wrote ../data/Beauty_and_Personal_Care/Beauty_and_Personal_Care.test.inter (656,647 rows)


## Analyze user interactions by split

In [11]:
train_counts = df_reviews_train.groupby("user_id", sort=False).size().rename("num_train")
valid_counts = df_reviews_valid.groupby("user_id", sort=False).size().rename("num_valid")
test_counts = df_reviews_test.groupby("user_id", sort=False).size().rename("num_test")

df_user_interactions = (
    pd.concat([train_counts, valid_counts, test_counts], axis=1)
    .fillna(0)
    .astype(int)
    .reset_index()
)

print("Sample of user interactions:")
display(df_user_interactions.sample(20))
print(f"\nTotal unique users: {len(df_user_interactions):,}")

Sample of user interactions:


,user_id,num_train,num_valid,num_test
201139,AFRNS7O77BFJS5HPRBNQ3T23JRRA,1,0,0
4369886,AH3DS5PIZK4PXPIFWZFMLPYIRKUA,1,0,0
2365530,AHYWD2M6JAFBTNYI3TSLOJBJMUGQ,1,0,0
5363535,AFCGHVFMCQPUFDWRYXYFO2NYJN6A,0,1,0
302586,AEPODN2WTVUBEJWB3JUOW7MZOSEQ,1,0,0
1029463,AFH6OI2SH5PDLSDKOC4OEU5CI7OQ,1,0,0
3256548,AEAURW5MX6CPPXAP2MX3ZC5RZWSA,1,0,0
4171819,AGHRUP5QKUZFD45MKBNVT4KBJFAQ,1,0,0
363212,AHXXILM4VZAKND6JPJBJONSEP2RQ,1,1,0
3574529,AEFCWZJ4TFNQFEYA5YCLJBR4ATKQ,1,0,0



Total unique users: 6,150,175


## Define cold vs. warm users with train data

In [12]:
df_train_users = df_user_interactions[df_user_interactions["num_train"] > 0]
cold_user_ids: set[str] = set(df_train_users[df_train_users["num_train"] < WARM_USER_MIN_REVIEWS]["user_id"])
warm_user_ids: set[str] = set(df_train_users[df_train_users["num_train"] >= WARM_USER_MIN_REVIEWS]["user_id"])
print(f"Warm users (>= {WARM_USER_MIN_REVIEWS} reviews): {len(warm_user_ids)}")
print(f"Cold users (< {WARM_USER_MIN_REVIEWS} reviews): {len(cold_user_ids)}")

Warm users (>= 5 reviews): 215239
Cold users (< 5 reviews): 5010220


## Write .user for user categories

In [13]:
def get_user_category(user_id: str) -> int:
    if user_id in warm_user_ids:
        return 0 # Warm user
    elif user_id in cold_user_ids:
        return 1 # Cold user
    else:
        return 2 # New user

def write_user_file(path: Path, rows: Iterable[dict[str, str | int | float]]) -> None:
    out_dir.mkdir(parents=True, exist_ok=True)
    row_count: int = 0

    with path.open("w", encoding="utf-8") as f:
        f.write(
            "user_id:token\t"
            "category:token\n"
        )

        for row in rows:
            f.write(
                f"{row['user_id']}\t"
                f"{row['category']}\n"
            )
            row_count += 1

    print(f"Wrote {path} ({row_count:,} rows)")


write_user_file(prefix.with_suffix(".user"), [
    {
        "user_id": user_map[user_id],
        "category": get_user_category(user_id),
    }
    for user_id in df_user_interactions["user_id"]
])


Wrote ../data/Beauty_and_Personal_Care/Beauty_and_Personal_Care.user (6,150,175 rows)
